In [43]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [45]:
from src.trainer import AGEMTrainer, IntervalAGEMTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

from configs import MNIST_AGEM_CONFIG as CONFIG

### Task Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)


In [ ]:
agem_trainer = AGEMTrainer(
    model,
    memory_samples=100,
    paradigm="TIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        context_id=i,
    )
    agem_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

### Domain Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=2, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
agem_trainer = AGEMTrainer(
    model,
    memory_samples=100,
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    agem_trainer.test(test_tasks)

### Class Incremental Training

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
agem_trainer = AGEMTrainer(
    model,
    memory_samples=200,
    paradigm="CIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    agem_trainer.test(test_tasks)

## IntervalAGEMTrainer

### Class Incremental Learning

In [4]:
from configs import MNIST_IT_CONFIG as INTERVAL_CONFIG

In [5]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


In [7]:
agem_trainer = IntervalAGEMTrainer(
    model,
    memory_samples=200,
    checkpoint=INTERVAL_CONFIG["checkpoint"],
    n_iters=INTERVAL_CONFIG["n_iters"],
    min_acc_limit=INTERVAL_CONFIG["min_acc_limit"],
    min_acc_increment=INTERVAL_CONFIG["min_acc_increment"],
    primal_learning_rate=INTERVAL_CONFIG["primal_learning_rate"],
    dual_learning_rate=INTERVAL_CONFIG["dual_learning_rate"],
    projection_strategy=INTERVAL_CONFIG["projection_strategy"],
    n_certificate_samples=INTERVAL_CONFIG["n_certificate_samples"],
    penalty_coefficient=INTERVAL_CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    agem_trainer.test(test_tasks)
    if i < len(test_tasks) - 1:
        agem_trainer.compute_rashomon_set(test)

Training Epochs: 100%|██████████████████████████████████████| 5/5 [00:07<00:00,  1.41s/it, loss=0.0033, acc=1.0000]


Test Results: [(0.0358, 0.9865), (46.3004, 0.0), (39.2716, 0.0), (47.2765, 0.0), (43.7228, 0.0)] (Avg: (35.3214, 0.1973))
Initial acc constraint violation: -0.1251 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.86
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|███████████████████████████████| 200/200 [00:10<00:00, 19.71it/s, size=1146.76, obj=0.185, min_soft_acc=1.000]


Final bbox:  Obj=0.19,  Size=1146.76,  Min acc hard=0.89,  Min acc soft=0.89
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['28.07', '135.35', '324.35', '455.03', '576.88', '690.13', '804.01', '919.52', '1035.06', '1146.76']
Checkpoint certificates: ['0.90', '0.88', '0.87', '0.84', '0.85', '0.88', '0.89', '0.88', '0.88', '0.89']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|██████████████████████████████████████| 5/5 [00:18<00:00,  3.74s/it, loss=3.1796, acc=0.1795]


...

Test Results: [(0.1464, 0.961), (11.9345, 0.1518), (6.3886, 0.144), (4.0386, 0.0366), (43.9931, 0.0)] (Avg: (13.3002, 0.2587))
Initial acc constraint violation: -0.0129 (Positive = violated)
Computing Rashomon set within outer box of size: 577.45
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.02
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.03,  Min acc soft=0.03


100%|████████████████████████████████| 200/200 [00:10<00:00, 19.35it/s, size=199.52, obj=0.032, min_soft_acc=0.015]


Final bbox:  Obj=0.03,  Size=199.52,  Min acc hard=0.00,  Min acc soft=0.01
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['11.45', '28.72', '46.94', '65.48', '84.04', '103.76', '124.86', '146.95', '173.67', '199.52']
Checkpoint certificates: ['0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.01', '0.00', '0.00']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|██████████████████████████████████████| 5/5 [00:21<00:00,  4.33s/it, loss=9.0910, acc=0.0000]


Test Results: [(0.1355, 0.961), (17.4453, 0.1528), (11.6389, 0.1477), (11.0159, 0.0218), (9.7028, 0.0)] (Avg: (9.9877, 0.2567))


### Ablation Study

In [55]:
import wandb

In [ ]:
for paradigm in ["TIL", "DIL", "CIL"]:
    for i in range(5):
        with wandb.init(
            project="certified-continual-learning",
            reinit=True,
            tags=["final_mnist_buffer", "buffer_agem", f"buffer_{paradigm.lower()}"],
        ):
            def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
                """Map the global label to the in context label."""
                return labels % 2
            wandb.log({"seed": i})
            SEED = i
            train_tasks, _, test_tasks = get_mnist_tasks(seed=SEED, train_val_split_ratio=1)
            context_sets = get_context_sets(test_tasks)
            head = InContextHead(context_sets, 10, device="cuda")
            head.set_context(0)
            model = models.get_mnist_model(device="cuda", output_dim=2 if paradigm == "DIL" else 10, seed=SEED, head = head if paradigm=="TIL" else None)
            print(
                f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
            )

            agem_trainer = AGEMTrainer(
                model,
                memory_samples=250,
                paradigm=paradigm,
                seed=SEED,
                domain_map_fn=domain_map_fn if paradigm == "DIL" else None
            )

            acc_matrix = []
            for i, (train, test) in enumerate(
                zip(train_tasks, test_tasks)
            ):
                agem_trainer.train(
                    train,
                    test,
                    batch_size=CONFIG["batch_size"],
                    epochs=CONFIG["epochs"],
                    lr=CONFIG["lr"],
                    weight_decay=CONFIG["weight_decay"],
                    context_id=i if paradigm == "TIL" else None,
                    val_freq=200
                )
                results = agem_trainer.test(
                    test_tasks,
                    context_list=list(range(len(test_tasks)))
                    if paradigm == "TIL"
                    else [None] * len(test_tasks),
                )
                accs = [res[1] for res in results]
                acc_matrix.append(accs)

            columns = [f"Test Task {i}" for i in range(len(test_tasks))]
            rows = [f"Task {i}" for i in range(len(test_tasks))]
            wandb.log(
                {
                    "accuracy_matrix": wandb.Table(
                        data=acc_matrix, columns=columns, rows=rows
                    )
                }
            )
            wandb.finish()

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


Training Epochs: 100%|██████████████████████████████████████████████████████████████████████████████████| 5/5 [00:11<00:00,  2.36s/it, loss=0.0009, acc=None]


Test Results: [(0.019, 0.9935), (0.6806, 0.7722), (1.0577, 0.5851), (3.0035, 0.4802), (4.9309, 0.3362)] (Avg: (1.9383, 0.6334))


Training Epochs: 100%|██████████████████████████████████████████████████████████████████████████████████| 5/5 [00:30<00:00,  6.16s/it, loss=0.0017, acc=None]


Test Results: [(0.1929, 0.9401), (0.0343, 0.9899), (1.0632, 0.5801), (1.4811, 0.6626), (6.8977, 0.1601)] (Avg: (1.9338, 0.6666))


...

Training Epochs: 100%|██████████████████████████████████████████████████████████████████████████████████| 5/5 [00:32<00:00,  6.41s/it, loss=0.0977, acc=None]


Test Results: [(0.0943, 0.9661), (0.0454, 0.9829), (0.6468, 0.7519), (0.467, 0.8171), (2.2689, 0.4393)] (Avg: (0.7045, 0.7915))


Training Epochs: 100%|██████████████████████████████████████████████████████████████████████████████████| 5/5 [00:48<00:00,  9.68s/it, loss=0.0068, acc=None]


Test Results: [(0.1709, 0.9487), (0.2469, 0.9239), (0.0534, 0.9769), (0.1078, 0.966), (4.47, 0.3013)] (Avg: (1.0098, 0.8234))


Training Epochs:  80%|█████████████████████████████████████████████████████████████████▌                | 4/5 [00:47<00:12, 12.06s/it, loss=0.0038, acc=None]